In [ ]:
import numpy as np, matplotlib.pyplot as plt
from scipy.special import sici

This notebook calculates make up gain at common sample rates for `hart::Sawtooth` signal.

In [ ]:
def sawtooth_blep(frequency_hz, buffer):
    phase_rad = 0
    phase_increment_rad = 2 * np.pi * frequency_hz / sample_rate_hz
    ratio = frequency_hz / sample_rate_hz

    for frame in range(buffer.size):
        portion = phase_rad / (2 * np.pi)
        value = 2 * portion - 1

        if portion < ratio:
            t = portion / ratio
            value += (t - 1) * (t - 1)
        elif portion > 1.0 - ratio:
            t = (portion - 1) / ratio
            value -= (t + 1) * (t + 1)

        buffer[frame] = value

        phase_rad = (phase_rad + phase_increment_rad) % (2 * np.pi)

In [ ]:
sample_rates_hz = [44100, 48000, 88200, 96000, 192000]
freqs_hz = np.arange(1, 20002, 601)
cycles_to_generate = 50

observed_peaks_linear = {}

for sample_rate_hz in sample_rates_hz:
    observed_peaks_linear[sample_rate_hz] = np.empty_like(freqs_hz, dtype=float)

    for i in range(freqs_hz.size):
        buffer_audio = np.empty(round(cycles_to_generate / freqs_hz[i] * sample_rate_hz))
        sawtooth_blep(freqs_hz[i], buffer_audio)
        observed_peaks_linear[sample_rate_hz][i] = np.abs(buffer_audio).max()

In [ ]:
plt.figure(figsize=(12,8))
a0s = []
a1s = []
a2s = []
approx_peaks_linear = {}
legend = []
for sample_rate_hz in observed_peaks_linear:
    a2, a1, a0 = np.polyfit(freqs_hz, observed_peaks_linear[sample_rate_hz], 2)
    approx_peaks_linear[sample_rate_hz] = freqs_hz * (a2 * freqs_hz + a1) + a0

    plt.scatter(freqs_hz, observed_peaks_linear[sample_rate_hz])
    plt.plot(freqs_hz, approx_peaks_linear[sample_rate_hz])

    a0s.append(a0)
    a1s.append(a1)
    a2s.append(a2)

    legend += [f'{sample_rate_hz} observed', f'{sample_rate_hz} approx']
plt.ylim(0, 1.1)
plt.legend(legend)
plt.xlabel('Sawtooth freq Hz')
plt.ylabel('Sample peak linear')
plt.savefig('Sawtooth peaks.png')

In [ ]:
for a0, a1, a2, freq_hz in zip(a0s, a1s, a2s, observed_peaks_linear):
    print(f'{freq_hz}: {a0:0.16}, {a1:0.16}, {a2:0.16}')